# 00 - Setup

First notebook in the pipeline — run once per fresh Colab runtime, **before**
`01_preprocessing`, `02_run_methods`, and `03_results_analysis`. It:

1. installs the Python packages this repo needs (`requirements.txt`),
2. pulls the external reference repos (`external/*`, git submodules) if they're missing,
3. creates the working folders (`data/raw`, `data/processed`, `output`),
4. (optional) mounts Google Drive and points `DATA_ROOT` / `OUTPUT_ROOT` at a persistent
   Drive folder so data and results survive runtime restarts.

Run against a **Google Colab** kernel connected through the VS Code Colab extension: cells
execute on the remote runtime while working directly against your local workspace files — there
is **no `git clone` of this repo**. Each Colab runtime is ephemeral, so re-run this notebook
after a disconnect.

## Setup — run this one cell

In [ ]:
# ============================================================
# SETUP: dependencies + external repos + working folders
# ============================================================
import os, subprocess, pathlib

# Make sure we run from the repo root (the notebook lives in src/notebooks/).
root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("Repo root:", root, "\n")

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

# 1. Python dependencies -------------------------------------------------
# Colab preinstalls numpy/scipy/scikit-learn/Pillow/scikit-image/torch/torchvision;
# this mainly adds SimpleITK, h5py, opencv-python, einops, timm, thop. Idempotent.
sh("pip install -q -r requirements.txt")

# 2. External reference repos --------------------------------------------
# Git submodules pinned to specific commits. On a fresh clone these folders are
# empty; pull them here. Skipped when already present (e.g. synced by VS Code).
externals = {
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning": (
        "https://github.com/ZhangHH233/Retinal_OCT_Image_Segmentation_via_Deep_Learning.git",
        "ac6d4c5d263ce03b1bc9235b471a88f9dbb0d994",
    ),
    "external/Public-available-retinal-OCT-datasets": (
        "https://github.com/ZhangHH233/Public-available-retinal-OCT-datasets.git",
        "f50b6a390a1211f06bd34f244c8e53dfc110a266",
    ),
}
for path, (url, commit) in externals.items():
    if any(pathlib.Path(path).glob("*")):
        print(f"OK   {path} (already present)")
        continue
    try:
        sh(f"git submodule update --init -- {path}")
    except subprocess.CalledProcessError:
        # Not a git checkout (e.g. plain download) — clone at the pinned commit.
        sh(f"git clone {url} {path}")
        sh(f"git -C {path} checkout {commit}")

# 3. Working folders (gitignored) ----------------------------------------
# Local scratch on the runtime. If you mount Drive below, DATA_ROOT/OUTPUT_ROOT
# point at Drive instead and these local dirs just act as a fallback.
for d in ["data/raw", "data/processed", "output"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
    print(f"OK   {d}/")

print("\nSetup complete — run the verification cell below.")

## Verify environment

Confirms the runtime, key imports, and repo layout. If anything reports `MISSING`, re-run Setup.

In [ ]:
import sys, os
print("Python:", sys.version.split()[0])
!nvidia-smi -L || echo "No GPU detected — set Runtime > Change runtime type > GPU"

import numpy, scipy, skimage, sklearn, PIL, h5py, imageio, cv2, SimpleITK, matplotlib
import torch, torchvision, einops, timm, thop
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__, "| SimpleITK:", SimpleITK.Version())

print("\nRepo layout:")
for path in [
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning/SOTAS",
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning/Metrics",
    "external/Public-available-retinal-OCT-datasets",
    "data/raw", "data/processed", "output",
]:
    print(f"  {'OK     ' if os.path.isdir(path) else 'MISSING'} {path}")

## (Optional) Mount Google Drive for persistent data + outputs

`data/` and `output/` are created on the **ephemeral** Colab runtime, so they vanish when it
disconnects. Run this cell to mount Drive and point `DATA_ROOT` / `OUTPUT_ROOT` at a persistent
Drive folder (`MyDrive/Segmentation/`). Datasets and results then survive restarts, and anything
written under `OUTPUT_ROOT` can later be pulled to your local machine — and cleared off Drive —
with `scripts/pull_from_drive.py` (one-time setup: `scripts/setup_rclone.sh` / `.ps1`).

Downstream notebooks (`01`–`03`) read `DATA_ROOT` and `OUTPUT_ROOT`, so run this cell before them.
Skip it only if you're handling storage another way (then define those two variables yourself).

In [ ]:
# Mount Drive and point the working roots at a persistent Drive folder, so
# preprocessed data and outputs survive Colab restarts — and so outputs can be
# pulled to your machine later with scripts/pull_from_drive.py.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT  = Path('/content/drive/MyDrive/Segmentation')
DATA_ROOT   = DRIVE_ROOT / 'data'      # raw/ + processed/ datasets
OUTPUT_ROOT = DRIVE_ROOT / 'output'    # metric CSVs, overlays, checkpoints

# Create the Drive-side folders (safe to re-run). The pull script defaults to
# pulling this OUTPUT_ROOT, so keep the 'Segmentation/output' path in sync with
# scripts/pull_from_drive.py's --drive-base / subpath.
for d in [DATA_ROOT / 'raw', DATA_ROOT / 'processed', OUTPUT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)